# SmolLM2 safetensors on PYNQ-Z1

The host exported `HuggingFaceTB/SmolLM2-135M-Instruct` through PyTorch
(transformers, our torchao W8A8 config, `torch.export`, torch-mlir) and
compiled it with `pynqo-backend`. This notebook loads that package with
`src/runtime`: CPU code runs on the Cortex-A9, every INT8 contraction runs on
the NPU through `__delegate__` and libnpu_rt. Run all cells in the root PYNQ
Python kernel; change the prompt in the generation cell.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path(os.environ.get("NPU_SMOLLM_DEMO", ".")).resolve()
sys.path.insert(0, str(ROOT))
print("bundle:", ROOT)

## Load the overlay and the compiled package

The first load builds libnpu_rt and the delegate dispatcher with the board's
own compiler (about half a minute); later loads reuse them.

In [ ]:
from pynq import Overlay
from src.runtime.tasks.llm import ExportedLLM

overlay = Overlay(str(ROOT / "overlay" / "npu_matrix.bit"))
llm = ExportedLLM(ROOT / "package", overlay)
print("NPU delegates:", len(llm.package.manifest["delegates"]))

## Generate

In [ ]:
prompt = "What is the capital of France?"
max_new_tokens = 32
ids, text, stats = llm.generate(
    prompt, max_new_tokens=max_new_tokens, chat=True,
    on_token=lambda _, piece: print(piece, end="", flush=True),
)
print()
print(json.dumps({"output_ids": ids, **stats}, indent=2))

## Release board buffers

Run after generation, or before loading the package again.

In [ ]:
llm.close()
del llm
print("Board buffers released")